# 1 · The rules, the gold set, and the verifiers — free

**No API key, no network, no spend.** Everything here reads the repository's own
declarations and runs its offline checks.

Start here even if you have a key. If this notebook runs clean, your checkout is
sound; if it does not, nothing you measure later means anything.

> **The rule for every notebook in this directory: they import from `loopeng` and
> contain no loop logic.** A notebook cell is the easiest place in the world to paste
> something that works, and a second implementation of the retry policy living in a
> cell is the exact defect this whole project is about. `tests/test_notebooks.py`
> enforces it.

In [ ]:
import os
from pathlib import Path

# Jupyter starts the kernel in the notebook's own directory, and
# everything in this repository is addressed from the root: `.env`,
# the warehouse, `gold/`, `results/`. Move there once. Idempotent, so
# re-running the cell is a no-op.
if not Path("pyproject.toml").exists():
    os.chdir("..")
print("working from", Path.cwd().name)


## The one place business rules are declared

In [ ]:
from loopeng.warehouse.schema import load_semantic_model

model = load_semantic_model()
print(f"{len(model['rules'])} rules, declared in semantic_model.yaml\n")
for name, rule in model["rules"].items():
    print(f"[{name}] applies to {', '.join(rule['applies_to'])}")
    print(f"    {' '.join(rule['statement'].split())}\n")

The conversion factors live in the same file, which is what stops a rate being typed
into three modules and edited in one.

In [ ]:
model["usd_factor"]

## The two prompt levels — the session's headline

`L0` withholds the rules. `L3` supplies them. Same model, same items, same loop: the
only thing that changes is whether the rules are in the prompt.

In [ ]:
from loopeng.prompts import LEVELS, render_prompt

for level in LEVELS:
    body = render_prompt(level)
    print(f"{level}: {len(body):,} characters")

In [ ]:
# The difference, in the model's own words. L0 gets the schema and nothing else.
print(render_prompt("L0")[-600:])

## The rule surface: what a verifier is actually checked against

Two probes per rule — one query that **breaks** it and must be rejected, one that is
**correct but unusual** and must be accepted. The second is what stops a verifier
scoring perfectly by rejecting everything.

`n_declared_rules` is reported beside `n_rules` so the denominator cannot quietly
shrink: counting probes alone reads as full coverage however many rules exist, which is
exactly how one rule went unprobed while the output said 6/6.


In [ ]:
from loopeng.verify.probes import run_probes

report = run_probes()
print(f"{report['n_sound']} of {report['n_rules']} probed rules sound")
print(f"{report['n_missed_violations']} missed violation(s), "
      f"{report['n_false_rejections']} false rejection(s)\n")
for rule, result in report["by_rule"].items():
    mark = "ok  " if result["sound"] else "FAIL"
    print(f"{mark} {rule}")


## The gold set

Built from parameterised SQL patterns and executed against a seeded warehouse, so
every answer is a fact about the data rather than an opinion about it.

In [ ]:
from pathlib import Path

from loopeng.gold.build import read_gold, split_items

items = read_gold(Path("gold/gold.jsonl"))
held_out, development = split_items(items)
print(f"{len(items)} items · {len(held_out)} held out · {len(development)} development")
print(f"{len({i.pattern_key for i in items})} patterns")

In [ ]:
item = held_out[0]
print(item.question)
print()
print(item.gold_sql)
print()
print("rules that apply:", ", ".join(item.rules) or "(none)")

**Items are clustered, not independent.** Each pattern contributes several items, so a
flaw in one pattern fails them together — which is why every interval this project
draws is narrower than the evidence strictly supports, and why every caption says so.

In [ ]:
from collections import Counter

for pattern, n in sorted(Counter(i.pattern_key for i in items).items()):
    print(f"{n:>3}  {pattern}")